# Example: Short-Time FFT Spectrogram with `sid.spectrogram`

This example demonstrates `sid.spectrogram` for time-frequency analysis of
signals. It computes the STFT and visualises how frequency content
evolves over time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sid

%matplotlib inline

## Chirp Signal: Frequency Sweep

A signal whose instantaneous frequency increases linearly from 50 Hz
to 150 Hz over 5 seconds.

In [ ]:
Fs = 1000
Ts = 1 / Fs
N = 5000
t = np.arange(N) * Ts
f0, f1 = 50, 150
x_chirp = np.cos(2 * np.pi * (f0 + (f1 - f0) / (2 * t[-1]) * t) * t)

result = sid.spectrogram(x_chirp, window_length=256, sample_time=Ts)

fig, ax = plt.subplots()
sid.spectrogram_plot(result, ax=ax)
ax.set_title('Chirp Signal: 50 Hz to 150 Hz')
plt.tight_layout()
plt.show()

## Window Length Trade-off

Short window: good time resolution, poor frequency resolution.  
Long window: poor time resolution, good frequency resolution.

In [ ]:
r_short = sid.spectrogram(x_chirp, window_length=64, sample_time=Ts)
r_long = sid.spectrogram(x_chirp, window_length=512, sample_time=Ts)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sid.spectrogram_plot(r_short, ax=axes[0])
axes[0].set_title('Short Window (L=64)')
sid.spectrogram_plot(r_long, ax=axes[1])
axes[1].set_title('Long Window (L=512)')
plt.tight_layout()
plt.show()

## Window Types

Different windows affect the trade-off between main lobe width and
side lobe suppression.

In [ ]:
r_hann = sid.spectrogram(x_chirp, window_length=256, window='hann', sample_time=Ts)
r_hamm = sid.spectrogram(x_chirp, window_length=256, window='hamming', sample_time=Ts)
r_rect = sid.spectrogram(x_chirp, window_length=256, window='rect', sample_time=Ts)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sid.spectrogram_plot(r_hann, ax=axes[0])
axes[0].set_title('Hann Window')
sid.spectrogram_plot(r_hamm, ax=axes[1])
axes[1].set_title('Hamming Window')
sid.spectrogram_plot(r_rect, ax=axes[2])
axes[2].set_title('Rectangular Window')
plt.tight_layout()
plt.show()

## Multi-Channel Signal

Two channels: channel 1 has a chirp, channel 2 has a fixed 100 Hz sinusoid.

In [ ]:
x_fixed = np.cos(2 * np.pi * 100 * t)  # 100 Hz constant tone
x_mc = np.column_stack([x_chirp, x_fixed])

result_mc = sid.spectrogram(x_mc, window_length=256, sample_time=Ts)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sid.spectrogram_plot(result_mc, channel=0, ax=axes[0])
axes[0].set_title('Channel 1: Chirp')
sid.spectrogram_plot(result_mc, channel=1, ax=axes[1])
axes[1].set_title('Channel 2: Fixed 100 Hz')
plt.tight_layout()
plt.show()

## Log Frequency Scale and NFFT Zero-Padding

In [ ]:
r_zp = sid.spectrogram(x_chirp, window_length=128, nfft=1024, sample_time=Ts)

fig, ax = plt.subplots()
sid.spectrogram_plot(r_zp, frequency_scale='log', ax=ax)
ax.set_title('Log Frequency Scale with Zero-Padding (NFFT=1024)')
plt.tight_layout()
plt.show()

## Accessing Raw STFT Data

The result object contains `power` (one-sided PSD), `power_db` (in dB),
and `complex_stft` (STFT coefficients).

In [ ]:
print('Spectrogram dimensions:')
print(f'  Time points:      {len(result.time)}')
print(f'  Frequency bins:   {len(result.frequency)}')
print(f'  Power shape:      {result.power.shape}')
print(f'  Complex shape:    {result.complex_stft.shape}')